# Skill Training Example

This notebook allows you to:
1. Connect to the R2 Robot backend, which includes a training endpoint.
2. Start training jobs.
3. Monitor training progress.
4. Cancel training if needed.

Uses the `TrainerClient` from the main SDK client module (`r2_labs.sdk.client`).

Remember to start the R2 backend first before running this notebook.

In [ ]:
from r2_labs.rpc import client as rpc_client
from r2_labs.sdk import client as sdk_client
from r2_labs.sdk import rpc_api
from r2_labs.sdk.progress_widget import (
    SkillTrainingProgressWidget,
    monitor_skill_training,
)

%load_ext autoreload
%autoreload 2

## 1. Configure Server Connection

Set the hostname/IP of the machine running the R2 backend.

In [ ]:
# Configure the server address
TRAINING_SERVER_HOST = "localhost"  # Change to your server IP/hostname
TRAINING_SERVER_PORT = rpc_api.DEFAULT_MODEL_TRAINER_PORT  # 7534

server_address = f"tcp://{TRAINING_SERVER_HOST}:{TRAINING_SERVER_PORT}"
print(f"Training server: {server_address}")

## 2. Connect to the Server

Create a `TrainerClient` using the SDK client module.

In [ ]:
# Create the base RPC client
base_client = rpc_client.BaseClient(
    server_address,
    timeout=30000,  # 30 second timeout
)

# Create the trainer client from the SDK
trainer = sdk_client.TrainerClient(base_client)

# Test connection by getting status
try:
    status = trainer.get_training_status()
    print("Connected to Training server!")
except Exception as e:
    print(f"Failed to connect: {e}")

## 4. Start Training

### Option A: Use entry_filter (auto-build the dataset from the R2 Data Warehouse)

In [ ]:
# Training configuration with entry filter
MODEL_NAME = "rectify_open_latch"
TRAINING_STEPS = 40000
ENTRY_FILTER = "rectify_open_latch*"  # Glob pattern to match data warehouse entries

# Start training with entry_filter
response = trainer.train_skill_model(
    model_name=MODEL_NAME,
    training_steps=TRAINING_STEPS,
    entry_filter=ENTRY_FILTER,
    batch_size=32,
    prediction_horizon=32,
    force_rebuild=False,  # Set to True to force rebuild the dataset
    timeout=600000,  # 10 minute timeout to give time for dataset rebuild.
)

if response.error:
    print(f"ERROR: {response.error}")
else:
    print("Training process started!")
    print("Use get_training_status() or monitor_training() to track progress.")

## 3. Check Current Training Status

Use this to check if training is already running.

In [ ]:
def print_status(status):
    """Pretty print training status."""
    print("=" * 50)
    phase = getattr(status, 'phase', 'unknown')
    print(f"Phase: {phase}")
    print(f"Is Finished: {status.is_finished}")
    if phase == "exporting":
        export_processed = getattr(status, 'export_entries_processed', 0)
        export_total = getattr(status, 'export_entries_total', 0)
        print(f"Export Progress: {export_processed} / {export_total}")
    else:
        print(f"Steps: {status.steps_completed} / {status.max_steps}")
        if status.loss is not None:
            print(f"Loss: {status.loss:.6f}")
        if status.fps is not None:
            print(f"Speed: {status.fps:.2f} steps/sec")
        if status.seconds_per_step is not None:
            print(f"Time per step: {status.seconds_per_step:.3f} sec")
        if status.metrics:
            print("Metrics:", status.metrics)
    print("=" * 50)

# Check current status
status = trainer.get_training_status()
print_status(status)

## 5. Monitor Training Progress

Use the `SkillTrainingProgressWidget` to monitor training with a clean UI.
Two options:
- **Option A**: Use `monitor_skill_training()` for a simple blocking call
- **Option B**: Use `SkillTrainingProgressWidget` directly for more control

In [ ]:
# Option A: Simple blocking call - waits until training completes
# final_status = monitor_skill_training(trainer, poll_interval=2.0)
# print(f"Final loss: {final_status.loss}")

# Option B: Widget with more control (interrupt kernel to stop early)
widget = SkillTrainingProgressWidget(trainer, poll_interval=2.0, color="blue")
widget.start()
widget.wait()  # Blocks until training completes

## 6. Cancel Training

Use this to stop training early if needed.

In [ ]:
# Cancel training (set export_model=True to export before cancelling)
response = trainer.cancel_training(export_model=False)

if response.success:
    print("Training cancelled successfully!")
    if response.model_id:
        print(f"Model ID: {response.model_id}")
else:
    print(f"Failed to cancel: {response.error}")

## 7. Export Model

Export the trained model to the model warehouse.

In [ ]:
# Export model after training
response = trainer.export_model()

if response.success:
    print("Model exported successfully!")
    print(f"Model ID: {response.model_id}")
else:
    print(f"Failed to export model: {response.error}")

## 8. One-time Status Check

In [ ]:
# Quick status check
status = trainer.get_training_status()
print_status(status)